# Week 9: Transformers & LLM Architectures
## Notebook 12: Speculative Sampling: Rejection Criteria & Exact Distribution

### 1. Conceptual Intuition & Theoretical Background
Autoregressive decoding generates one token per forward pass, underutilizing modern parallel GPU hardware.

**Speculative Sampling (Leviathan et al., 2023; Chen et al., 2023):**
1. **Draft Generation:** A small, fast draft model $M_{\text{draft}}$ generates $K$ candidate tokens speculatively.
2. **Parallel Verification:** The large target model $M_{\text{target}}$ evaluates all $K$ candidate tokens in a **single forward pass**.
3. **Rejection Sampling Criterion:**
   Token $x_{n}$ proposed by draft distribution $q(x)$ is accepted by target distribution $p(x)$ with probability:
   $$P(\text{Accept}) = \min\left(1,\ \frac{p(x)}{q(x)}\right)$$
4. **Correction on Rejection:**
   If a token is rejected at index $k$, it is resampled from the adjusted distribution:
   $$p'(x) = \text{norm}\left( \max(0, p(x) - q(x)) \right)$$

**Exact Distribution Guarantee:**
The sampled output matches the true target distribution $p(x)$ **identically**, with zero degradation in generation quality.

---

### 2. Scratch Python Implementation: Speculative Sampling Loop

In [ ]:
import numpy as np

def speculative_sampling_step(target_logits, draft_logits, K_draft_tokens):
    """
    Evaluates speculative draft tokens against target distribution.
    Returns: Accepted tokens + 1 correction token.
    """
    def softmax(z):
        ez = np.exp(z - np.max(z))
        return ez / np.sum(ez)
        
    p_probs = [softmax(tl) for tl in target_logits]
    q_probs = [softmax(dl) for dl in draft_logits]
    
    accepted_tokens = []
    
    for i, token in enumerate(K_draft_tokens):
        p_val = p_probs[i][token]
        q_val = q_probs[i][token]
        
        # Acceptance probability: min(1, p/q)
        accept_prob = min(1.0, p_val / max(q_val, 1e-9))
        
        if np.random.rand() < accept_prob:
            accepted_tokens.append(token)
        else:
            # Rejection: Sample correction token from max(0, p - q)
            diff = np.maximum(0.0, p_probs[i] - q_probs[i])
            diff /= np.sum(diff) # Normalize
            correction_token = np.random.choice(len(diff), p=diff)
            accepted_tokens.append(correction_token)
            # Stop verification at first rejection
            return accepted_tokens, False
            
    # If all K accepted, sample bonus token from target model
    bonus_token = np.random.choice(len(p_probs[-1]), p=p_probs[-1])
    accepted_tokens.append(bonus_token)
    return accepted_tokens, True

# Benchmark: 3 Draft Tokens Verified by Target Model
vocab_size = 5
K = 3

np.random.seed(42)
# Draft model proposes tokens [1, 2, 4]
draft_tokens = [1, 2, 4]

# Simulated logits for target and draft models
mock_draft_logits = [np.random.randn(vocab_size) for _ in range(K)]
mock_target_logits = [dl + np.random.normal(0, 0.4, vocab_size) for dl in mock_draft_logits]
mock_target_logits.append(np.random.randn(vocab_size)) # Bonus step

accepted_seq, all_accepted = speculative_sampling_step(
    mock_target_logits, mock_draft_logits, draft_tokens
)

print("--- Speculative Sampling Execution ---")
print(f"Draft Tokens Proposed:      {draft_tokens}")
print(f"Tokens Generated in 1 Pass: {accepted_seq}")
print(f"Speedup Factor (Tokens/Fwd): {len(accepted_seq)}x")
print(f"All Proposed Tokens Passed: {all_accepted}")

### 3. Step-by-Step Code Walkthrough
1. **Parallel Verification:** Target model scores all $K$ tokens concurrently.
2. **Acceptance Threshold:** Accepts tokens based on the likelihood ratio $\frac{p(x)}{q(x)}$.
3. **Distribution Correction:** If rejected, samples from the normalized positive difference $\max(0, p - q)$.

---

### 4. Industry Applications & Trade-offs
- **Production Inference (vLLM, TensorRT-LLM):** Delivers $2-3\times$ speedups on latency-critical endpoints.
- **Trade-off:** Draft model must run on the same device or via fast shared memory to avoid inter-GPU communication latency.

---

### 5. Top Interview Questions & Answers
**Q1: Does Speculative Sampling change the output distribution of the language model?**
> **Answer:** No. Mathematical proofs (Leviathan et al.) demonstrate that sampling from $\min(1, p/q)$ and correcting with $\max(0, p - q)$ guarantees that the output matches the target distribution $p(x)$ exactly.

**Q2: What is the optimal draft length $K$ in Speculative Decoding?**
> **Answer:** Typically between $K=3$ and $K=5$. If $K$ is too large, the marginal acceptance probability decays exponentially ($\alpha^K$), wasting verification compute.